In [1]:
import pandas as pd

df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended.csv")
print(f"Loaded {len(df)} rows")
df.head()

Loaded 110000 rows


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,fa3e3f95-9200-41a8-a7d4-de04b6c986fc,James Ruesch,https://play-lh.googleusercontent.com/a-/ALV-U...,they deleted my wife's account without explana...,3,0,10.139,2026-07-23 02:30:32,"Hi James. Due to privacy regulations, we canno...",2026-07-23 02:56:22,10.139
1,e9e94068-5bd4-48fa-86e4-15f34ff6806d,Sand Summerstorm,https://play-lh.googleusercontent.com/a-/ALV-U...,not available in my region for no reason whats...,1,0,NaN,2026-07-23 01:15:47,NaN,NaN,NaN
2,a3476d6e-36f8-48a3-b4f2-56e5b3e4fd2b,Briege Hosford,https://play-lh.googleusercontent.com/a/ACg8oc...,Best app ever.,5,0,10.139,2026-07-23 00:06:07,NaN,NaN,10.139
3,acb8785b-a64c-4a7b-9a34-82c1f3f8b620,Andrew Freeman,https://play-lh.googleusercontent.com/a-/ALV-U...,App force closes every single time. I've clear...,1,0,NaN,2026-07-23 00:03:58,Hi. We understand your concern regarding the a...,2026-07-22 23:29:51,NaN
4,ff575420-a3e0-4ba4-a748-25fbe7be8364,Dylan Hodson,https://play-lh.googleusercontent.com/a-/ALV-U...,everytime apple trys to take money for apple m...,4,0,10.139,2026-07-23 00:03:35,NaN,NaN,10.139


In [2]:
df.isnull().sum()

reviewId                    0
userName                    3
userImage                   0
content                    10
score                       0
thumbsUpCount               0
reviewCreatedVersion    17870
at                          0
replyContent            92938
repliedAt               92938
appVersion              17870
dtype: int64

In [3]:
df["userName"] = df["userName"].fillna("Anonymous")
df["reviewCreatedVersion"] = df["reviewCreatedVersion"].fillna("Unknown")
df["appVersion"] = df["appVersion"].fillna("Unknown")
df["got_reply"] = df["replyContent"].notnull()

before = len(df)
df = df.dropna(subset=["content"])
df = df[df["content"].str.strip() != ""]
after = len(df)
print(f"Dropped {before - after} rows with empty review text")
print(f"Remaining rows: {after}")

Dropped 10 rows with empty review text
Remaining rows: 109990


## Remove duplicates

In [4]:
before = len(df)
df = df.drop_duplicates(subset=["reviewId"])
after = len(df)
print(f"Dropped {before - after} duplicate rows")

Dropped 0 duplicate rows


## Rename columns to match your original naming convention

In [5]:
df = df.rename(columns={
    "content": "review_text",
    "score": "rating",
    "at": "review_date",
    "thumbsUpCount": "thumbs_up",
})

df["review_date"] = pd.to_datetime(df["review_date"])
df["year"] = df["review_date"].dt.year
df["month"] = df["review_date"].dt.month
df["review_length"] = df["review_text"].str.len()

df[["review_date", "year", "rating"]].head()

,review_date,year,rating
0,2026-07-23 02:30:32,2026,3
1,2026-07-23 01:15:47,2026,1
2,2026-07-23 00:06:07,2026,5
3,2026-07-23 00:03:58,2026,1
4,2026-07-23 00:03:35,2026,4


## Sanity check

In [6]:
print(f"Final row count: {len(df)}")
print(f"\nRating distribution:")
print(df["rating"].value_counts().sort_index())
print(f"\nReviews per year:")
print(df["year"].value_counts().sort_index())

Final row count: 109990

Rating distribution:
rating
1    13873
2     1744
3     2199
4     8640
5    83534
Name: count, dtype: int64

Reviews per year:
year
2023    23322
2024    37821
2025    31607
2026    17240
Name: count, dtype: int64


## Save

In [7]:
df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended_clean.csv", index=False)
print("Saved.")

Saved.


## Extrat 2022 & 2021 

In [8]:
import pandas as pd
import time
from google_play_scraper import Sort, reviews

APP_ID = "com.revolut.revolut"
TARGET_COUNT = 300000
BATCH_SIZE = 200
SAVE_EVERY = 5000

all_reviews = []
continuation_token = None
last_save_count = 0

# NEW filename - avoids any conflict with existing files
output_path = "E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_2021_2026.csv"

while len(all_reviews) < TARGET_COUNT:
    try:
        result, continuation_token = reviews(
            APP_ID,
            lang="en",
            country="us",
            sort=Sort.NEWEST,
            count=BATCH_SIZE,
            continuation_token=continuation_token,
        )

        if not result:
            print("No more reviews available — stopping.")
            break

        all_reviews.extend(result)
        oldest_date = min(r['at'] for r in result if r.get('at'))
        print(f"Fetched {len(all_reviews)} total | Oldest in this batch: {oldest_date}")

        if len(all_reviews) - last_save_count >= SAVE_EVERY:
            try:
                df_temp = pd.DataFrame(all_reviews)
                df_temp.to_csv(output_path, index=False)
                last_save_count = len(all_reviews)
                print(f"  --> Progress saved ({len(all_reviews)} reviews)")
            except PermissionError:
                print("  --> Save skipped (file locked) - continuing fetch anyway")

        if continuation_token is None:
            print("Reached the end of available reviews.")
            break

        time.sleep(1)

    except Exception as e:
        print(f"Error occurred: {e}")
        break

df_final = pd.DataFrame(all_reviews)
df_final.to_csv(output_path, index=False)
print(f"\nDone. Total reviews saved: {len(df_final)}")
print(f"Date range: {df_final['at'].min()} to {df_final['at'].max()}")

Fetched 200 total | Oldest in this batch: 2026-07-20 14:09:18
Fetched 400 total | Oldest in this batch: 2026-07-16 20:04:52
Fetched 600 total | Oldest in this batch: 2026-07-13 21:17:03
Fetched 800 total | Oldest in this batch: 2026-07-10 23:24:26
Fetched 1000 total | Oldest in this batch: 2026-07-08 01:51:00
Fetched 1200 total | Oldest in this batch: 2026-07-03 22:58:28
Fetched 1400 total | Oldest in this batch: 2026-06-30 18:19:53
Fetched 1600 total | Oldest in this batch: 2026-06-26 12:19:57
Fetched 1800 total | Oldest in this batch: 2026-06-22 00:57:12
Fetched 2000 total | Oldest in this batch: 2026-06-18 16:59:05
Fetched 2200 total | Oldest in this batch: 2026-06-15 07:40:28
Fetched 2400 total | Oldest in this batch: 2026-06-11 01:05:54
Fetched 2600 total | Oldest in this batch: 2026-06-06 17:14:06
Fetched 2800 total | Oldest in this batch: 2026-06-01 23:02:22
Fetched 3000 total | Oldest in this batch: 2026-05-28 14:44:21
Fetched 3200 total | Oldest in this batch: 2026-05-23 10:23